In [1]:
import ollama 
import os
from tqdm import tqdm
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, confusion_matrix
import json
import signal
import argparse
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import copy
import numpy as np

import sys
from collections import defaultdict

In [2]:
sys.argv = [
    'notebook',  
    '--modelname', 'llama3.2-vision:90b', #'llama3.2-vision:90b',
    '--data', 'gully',
    '--test_data_path','/mnt/jacket/WACV-2025-Workshop-ViGIR/Testing_Set.json',
    '--subset', 'train',
    '--results_dir', '/mnt/jacket/WACV-2025-Workshop-ViGIR/results/proposed',
    '--timeout', '20',
    '--model_unloading'
]

In [3]:
parser = argparse.ArgumentParser(description="A script to evaluate V-LLMs on different image classification datasets")

parser.add_argument("--modelname", type=str, required=True, help="The name of the V-LLM model")
parser.add_argument("--data", type=str, required=True, help="Dataset name")
parser.add_argument("--test_data_path", type=str, required=True, help="Path to the image data dir")
parser.add_argument("--subset", type=str, required=True, help="train, test or validation set")
parser.add_argument("--results_dir", type=str, required=True, help="Folder name to save results")
parser.add_argument("--timeout", type=int, default=40, help="time out duration to skip one sample")
parser.add_argument("--model_unloading", action="store_true", help="Enables unloading mode. Every 100 sampels it unloades the model from the GPU to avoid carshing.")

args = parser.parse_args()

In [4]:
# Load test set:
with open(args.test_data_path, 'r') as file:
    test_data = json.load(file)

print('Number of Annotated GT Images: ', len(test_data.keys()))
test_data_keys = list(test_data.keys())
print('Number of Annotated GT Images (List): ', len(test_data_keys))

Number of Annotated GT Images:  311
Number of Annotated GT Images (List):  311


In [6]:
# Load results from a specifc model
#filename = 'reason_first_final_results_'+args.modelname+'.json'
#filename = 'results_all_'+args.modelname+'.json'
filename = 'results_18Q_'+args.modelname+'.json'
with open((os.path.join(args.results_dir, filename)), 'r') as file:
    result_data = json.load(file)

print('Results from: ', args.modelname)
print('Number of Images: ', len(result_data.keys()))
result_data_keys = list(result_data.keys())
print('Number of Images (List): ', len(result_data_keys))

Results from:  llama3.2-vision:90b
Number of Images:  311
Number of Images (List):  311


In [7]:
result_data

{'415': ['0'],
 '1020': ['0'],
 '105': ['0'],
 '439': ['0'],
 '914': ['4'],
 '1099': ['0'],
 '1065': ['0'],
 '373': ['0'],
 '166': ['0'],
 '396': ['0'],
 '837': ['0'],
 '685': ['4'],
 '226': ['4'],
 '956': ['0'],
 '70': ['0'],
 '604': ['0'],
 '1067': ['0'],
 '118': ['0'],
 '774': ['0'],
 '521': ['0'],
 '910': ['0'],
 '975': ['4'],
 '352': ['0'],
 '761': ['0'],
 '1007': ['0'],
 '428': ['0'],
 '1038': ['0'],
 '50': ['0'],
 '838': ['0'],
 '126': ['0'],
 '1078': ['4'],
 '944': ['0'],
 '532': ['0'],
 '1041': ['0'],
 '540': ['0'],
 '128': ['0'],
 '722': ['4'],
 '478': ['0'],
 '639': ['0'],
 '668': ['4'],
 '343': ['0'],
 '1021': ['0'],
 '552': ['0'],
 '848': ['0'],
 '74': ['0'],
 '520': ['0'],
 '1032': ['0'],
 '615': ['0'],
 '536': ['0'],
 '1117': ['0'],
 '646': ['0'],
 '390': ['0'],
 '923': ['0'],
 '194': ['4'],
 '216': ['0'],
 '99': ['0'],
 '372': ['0'],
 '857': ['0'],
 '335': ['4'],
 '505': ['0'],
 '972': ['0'],
 '727': ['0'],
 '1064': ['0'],
 '740': ['4'],
 '182': ['0'],
 '316': ['0'],
 '

In [8]:
test_data_list = [[int(key), int(value['label'])] for key, value in test_data.items()]
result_data_list = [[int(key), int(value[0])] for key, value in result_data.items()]

In [9]:
test_data_arr = np.array(test_data_list)
result_data_arr = np.array(result_data_list)

In [10]:
sorted_test_data_arr = test_data_arr[test_data_arr[:, 0].argsort()]
sorted_result_data_arr = result_data_arr[result_data_arr[:, 0].argsort()]

In [11]:
# Convert ground truth and predictions to dictionaries for easier comparison
ground_truth_dict = {int(key): int(value['label']) for key, value in test_data.items()}
predictions_dict = {int(key): int(value[0]) for key, value in result_data.items()}

In [12]:
y_true = np.array(list(ground_truth_dict.values()))/4
y_pred = np.array(list(predictions_dict.values()))/4
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')
f1_score_not = (2 * macro_f1) - f1

# Print the results
print(f"True Positives (TP): {tp}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Negatives (TN): {tn}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Accuracy: {accuracy:.2f}")
print(f"F1 Score (G): {f1:.2f}")
print(f"F1 Score (NG): {f1_score_not:.2f}")
print(f"Macro F1 Score: {macro_f1:.2f}")

True Positives (TP): 37
False Positives (FP): 10
False Negatives (FN): 140
True Negatives (TN): 124
Precision: 0.79
Recall: 0.21
Accuracy: 0.52
F1 Score (G): 0.33
F1 Score (NG): 0.62
Macro F1 Score: 0.48
